# Séparer les environnements de vecteurs — contrôle d'accès, pas commodité

*Recycler la documentation du projet LivresAgités (en sommeil) vers le dépôt
pédagogique. Ce notebook rend exécutable la leçon du **Parcours 2** du dossier
[`AI-Engine-WordPress`](./livresagites-parcours.md) : la séparation d'un vector
store en environnements distincts est un **contrôle d'accès**, pas un rangement
commode.*

> **Thèse.** Le réflexe est d'indexer « tout le site » dans un seul corpus.
> C'est une faille d'accès : du contenu réservé à un comité interne devient
> recherchable par n'importe quel visiteur. La séparation par environnement
> restreint ce que chaque chatbot peut atteindre — et son corollaire
> opérationnel est qu'une réindexation lancée sans préciser la cible écrase
> silencieusement un corpus voisin.

Ce notebook ne dépend d'aucun service : pas de modèle d'embeddings, pas de
vector store distant, pas de clé. La géométrie est **synthétique et
déterministe** — six clusters dans un espace de dimension 16, construits avec
`numpy` et une graine fixée. Ce qui est enseigné est la *structure* du défaut,
pas une mesure de performance sur un corpus réel.


## 1. Le problème en une phrase

Un chatbot public répond à partir d'un corpus indexé. Si ce corpus mélange le
catalogue public (livres en vente) et les manuscrits en cours d'évaluation
(comité interne), alors **n'importe quel visiteur peut, par une question bien
orientée, récupérer du contenu réservé**. Le filtre ne se répare pas dans le
prompt du chatbot (« ne parle pas des manuscrits ») : un modèle qui voit le
chunk dans son contexte s'appuie dessus. Le filtre se met **en dessous**, dans
le vector store, en séparant les corpus par environnement d'accès.


In [1]:
# dependances : numpy seulement. Aucun reseau, aucune cle.
import numpy as np

SEED = 7
DIM = 16
rng = np.random.default_rng(SEED)

# Six centres, un par environnement. En dimension 16, des vecteurs unitaires
# tires au hasard sont quasi orthogonaux : les clusters seront bien separes.
noms_env = [
    "catalogue_public",   # ce que voit un visiteur (livres en vente)
    "comite_lecture",     # manuscrits en cours d'evaluation (interne)
    "editorial",          # pipeline editorial (editrices)
    "auteurs",            # espace de travail des autrices
    "production",         # impression, stocks, calendriers
    "reference",          # documentation technique, consignes
]
centres = rng.standard_normal((len(noms_env), DIM))
centres /= np.linalg.norm(centres, axis=1, keepdims=True)

# Matrice des cosinus entre centres : on verifie qu'ils sont bien separes.
cos_centres = centres @ centres.T
hors_diag = cos_centres[~np.eye(len(noms_env), dtype=bool)]
print("Cosinus inter-centres (min/max hors diagonale) :")
print(f"  min = {hors_diag.min():+.3f}   max = {hors_diag.max():+.3f}")
print("  (proche de 0 = clusters orthogonaux = bien separes)")


Cosinus inter-centres (min/max hors diagonale) :
  min = -0.286   max = +0.329
  (proche de 0 = clusters orthogonaux = bien separes)


Les centres sont quasi orthogonaux (cosinus proche de zéro entre toute
paire). C'est la propriété qui rend la fuite **nette** : un chunk d'un
environnement n'est le voisin d'aucun chunk d'un autre environnement, sauf si
la requête vise délibérément cet environnement.


## 2. L'atelier — six environnements, six régimes d'accès

Chaque environnement regroupe des **chunks** (unités de corpus indexées). Un
chunk est un vecteur proche du centre de son environnement. Le régime d'accès
est porté par l'environnement, pas par le chunk : appartenir à
`comite_lecture` *est* le fait d'être réservé au comité.


In [2]:
def construire_store(centres, par_env=10, bruit=0.05, rng=None):
    """Cree un vector store partitionne en environnements.

    Renvoie un dict {nom_env: liste de (vecteur, etiquette)}.
    Chaque vecteur = centre + petit bruit, puis normalise.
    """
    store = {}
    for i, nom in enumerate(noms_env):
        chunks = []
        for k in range(par_env):
            v = centres[i] + bruit * rng.standard_normal(DIM)
            v /= np.linalg.norm(v)
            chunks.append((v, nom + "::chunk_" + format(k, "02d")))
        store[nom] = chunks
    return store


store = construire_store(centres, par_env=10, bruit=0.05, rng=rng)

# Releve : combien de vecteurs par environnement.
print("Vector store synthetique -- Maison Valmont")
print("-" * 48)
total = 0
for nom in noms_env:
    n = len(store[nom])
    total += n
    print("  " + nom.ljust(20) + str(n).rjust(3) + " vecteurs")
print("-" * 48)
print("  " + "TOTAL".ljust(20) + str(total).rjust(3) + " vecteurs (6 environnements)")


Vector store synthetique -- Maison Valmont
------------------------------------------------
  catalogue_public     10 vecteurs
  comite_lecture       10 vecteurs
  editorial            10 vecteurs
  auteurs              10 vecteurs
  production           10 vecteurs
  reference            10 vecteurs
------------------------------------------------
  TOTAL                60 vecteurs (6 environnements)


## 3. La requête naïve — sans filtre d'environnement

Un visiteur public demande où en est un manuscrit. La requête, une fois
embeddée, atterrit naturellement près du cluster `comite_lecture` (c'est le
sujet du chunk). Un retrieval qui ne filtre pas par environnement renvoie les
chunks les plus proches **tous régimes confondus**.


In [3]:
def plus_proches(store, requete, k=5, env_filtre=None):
    """Renvoie les k chunks les plus proches de la requete (cosinus).

    Si env_filtre est None : cherche dans TOUS les environnements.
    Sinon : ne cherche que dans l'environnement autorise.
    """
    cibles = store if env_filtre is None else {env_filtre: store[env_filtre]}
    scores = []
    rq = requete / np.linalg.norm(requete)
    for nom, chunks in cibles.items():
        for v, etiquette in chunks:
            scores.append((float(rq @ v), etiquette, nom))
    scores.sort(reverse=True)
    return scores[:k]


# La requete du visiteur : on l'embedde pres du cluster comite_lecture
# (elle porte sur les manuscrits en cours d'evaluation).
req_visiteur = centres[noms_env.index("comite_lecture")] + 0.04 * rng.standard_normal(DIM)

print("Requete : ou en est le manuscrit que j'ai soumis ?")
print("Utilisateur : visiteur public (autorise sur catalogue_public uniquement)")
print()
print("Top-5 SANS filtre d'environnement :")
for cos, etiquette, nom in plus_proches(store, req_visiteur, k=5, env_filtre=None):
    print("  " + format(cos, "+.3f") + "  [" + nom.ljust(18) + "]  " + etiquette)


Requete : ou en est le manuscrit que j'ai soumis ?
Utilisateur : visiteur public (autorise sur catalogue_public uniquement)

Top-5 SANS filtre d'environnement :
  +0.980  [comite_lecture    ]  comite_lecture::chunk_09
  +0.979  [comite_lecture    ]  comite_lecture::chunk_02
  +0.979  [comite_lecture    ]  comite_lecture::chunk_08
  +0.970  [comite_lecture    ]  comite_lecture::chunk_00
  +0.969  [comite_lecture    ]  comite_lecture::chunk_01


### Lecture du leak

Les cinq résultats viennent **tous** de `comite_lecture` — un environnement que
le visiteur n'est pas autorisé à voir. Le contenu réservé au comité interne est
remonté à un visiteur public, parce que rien dans le retrieval n'a dit « ne
cherche que dans `catalogue_public` ». **Le prompt du chatbot ne peut pas
réparer ça** : si le chunk est dans le contexte, le modèle s'appuie dessus.


## 4. La requête scopée — avec filtre d'environnement

Le même visiteur, la même requête, mais le retrieval est restreint à
l'environnement auquel il a accès : `catalogue_public`.


In [4]:
print("Requete : ou en est le manuscrit que j'ai soumis ?")
print("Utilisateur : visiteur public (autorise sur catalogue_public uniquement)")
print()
print("Top-5 AVEC filtre = catalogue_public :")
res = plus_proches(store, req_visiteur, k=5, env_filtre="catalogue_public")
for cos, etiquette, nom in res:
    print("  " + format(cos, "+.3f") + "  [" + nom.ljust(18) + "]  " + etiquette)
print()
print("Cosinus proches de 0 : le visiteur ne trouve rien de pertinent,")
print("et c'est exactement le comportement attendu.")


Requete : ou en est le manuscrit que j'ai soumis ?
Utilisateur : visiteur public (autorise sur catalogue_public uniquement)

Top-5 AVEC filtre = catalogue_public :
  +0.177  [catalogue_public  ]  catalogue_public::chunk_09
  +0.103  [catalogue_public  ]  catalogue_public::chunk_04
  +0.097  [catalogue_public  ]  catalogue_public::chunk_07
  +0.087  [catalogue_public  ]  catalogue_public::chunk_01
  +0.084  [catalogue_public  ]  catalogue_public::chunk_00

Cosinus proches de 0 : le visiteur ne trouve rien de pertinent,
et c'est exactement le comportement attendu.


### Lecture du bon comportement

La requête ne trouve rien de pertinent dans `catalogue_public` (les cosinus
sont bas, proches de zéro) — et c'est exactement ce qu'on veut. Le visiteur ne
voit pas le manuscrit, non pas parce qu'on a dit au modèle de se taire, mais
parce que **le chunk n'est pas dans son espace de recherche**. Le contrôle
d'accès est structural, pas discursif.


## 5. Mesurer le contraste — le taux de fuite

Une requête isolée ne prouve pas un défaut systémique. On mesure sur une
**batterie** : pour chaque environnement, deux requêtes dont l'embedding vise
ce cluster. Pour chacune, l'utilisateur est autorisé sur un environnement
**différent**. Le taux de fuite = fraction du top-5 qui provient d'un
environnement que l'utilisateur n'est pas censé voir.


In [5]:
def taux_de_fuite(store, requete, k, env_autorise):
    """Fraction du top-k provenant d'un environnement NON autorise."""
    rq = requete / np.linalg.norm(requete)
    res = plus_proches(store, rq, k=k, env_filtre=None)
    return sum(1 for _, _, nom in res if nom != env_autorise) / len(res)


# Batterie : 2 requetes par environnement, l'utilisateur etant autorise
# sur un AUTRE environnement (celui indexe +1, modulo).
K = 5
fuites_sans = []
fuites_avec = []
for i, nom in enumerate(noms_env):
    env_autorise = noms_env[(i + 1) % len(noms_env)]
    for _ in range(2):
        req = centres[i] + 0.04 * rng.standard_normal(DIM)
        fuites_sans.append(taux_de_fuite(store, req, K, env_autorise))
        fuites_avec.append(0.0)  # filtre = env_autorise -> 0 fuite par construction

sans = float(np.mean(fuites_sans))
avec = float(np.mean(fuites_avec))
print("Taux de fuite moyen sur " + str(len(fuites_sans)) + " requetes (top-" + str(K) + ") :")
print("  SANS filtre : " + format(sans, ".0%") + "  (les chunks fuient vers un utilisateur non autorise)")
print("  AVEC filtre : " + format(avec, ".0%") + "  (le retrieval est scope a l'autorise)")


Taux de fuite moyen sur 12 requetes (top-5) :
  SANS filtre : 100%  (les chunks fuient vers un utilisateur non autorise)
  AVEC filtre : 0%  (le retrieval est scope a l'autorise)


### Lecture de la mesure

Sans filtre, le taux de fuite est voisin de **100 %** : presque chaque requête
remonte des chunks d'un environnement que l'utilisateur n'est pas autorisé à
voir. Avec filtre, **0 %**. L'écart n'est pas une amélioration de qualité du
retrieval — c'est un **changement de régime d'accès**. La même métrique, sur le
même vector store, passe de la porte ouverte à la porte fermée selon qu'on
spécifie ou non l'environnement.


## 6. L'accident de réindexation — le corollaire silencieux

Le parcours documente un corollaire opérationnel appris à ses dépens :

> Une réindexation lancée sans préciser l'environnement cible écrase un corpus
> voisin.

On le simule. La fonction `reindexer` prend un nouveau corpus et un
environnement cible. Si la cible est `None`, elle écrit dans un emplacement par
défaut — qui, dans une implémentation négligente, **alias un environnement
existant**. La perte est immédiate et silencieuse.


In [6]:
def reindexer(store, nouveau_corpus, env_cible=None, defaut_aliase="comite_lecture"):
    """Reindexe un nouveau corpus dans un environnement.

    BUG : si env_cible est None, on ecrit dans un emplacement 'par defaut'
    qui, dans cette implementation negligente, alias un environnement voisin.
    La reindexation ecrase alors ce voisin, sans avertissement.
    """
    cible = env_cible if env_cible is not None else defaut_aliase
    store[cible] = [(v, etiquette) for v, etiquette in nouveau_corpus]
    return cible


# Avant : comite_lecture contient 10 manuscrits en cours d'evaluation.
avant = len(store["comite_lecture"])
print("Avant reindexation : comite_lecture = " + str(avant) + " vecteurs (manuscrits en evaluation)")

# On veut reindexer du NOUVEAU catalogue public (2 chunks de demonstration).
nouveau_catalogue = [
    (centres[noms_env.index("catalogue_public")] + 0.05 * rng.standard_normal(DIM),
     "catalogue_public::nouveau_0"),
    (centres[noms_env.index("catalogue_public")] + 0.05 * rng.standard_normal(DIM),
     "catalogue_public::nouveau_1"),
]

# BUG : l'environnement cible n'est pas precise.
cible_touchee = reindexer(store, nouveau_catalogue, env_cible=None)
apres = len(store["comite_lecture"])

print("reindexer(nouveau_catalogue, env_cible=None) -> ecrit dans '" + cible_touchee + "'")
print("Apres reindexation : comite_lecture = " + str(apres) + " vecteurs")
print()
if cible_touchee == "comite_lecture" and apres != avant:
    perte = avant - apres
    print(">>> PERTE SILENCIEUSE : " + str(perte) + " manuscrits du comite ecrases")
    print("    par du contenu catalogue. Aucun avertissement n'a ete emis.")


Avant reindexation : comite_lecture = 10 vecteurs (manuscrits en evaluation)
reindexer(nouveau_catalogue, env_cible=None) -> ecrit dans 'comite_lecture'
Apres reindexation : comite_lecture = 2 vecteurs

>>> PERTE SILENCIEUSE : 8 manuscrits du comite ecrases
    par du contenu catalogue. Aucun avertissement n'a ete emis.


### Lecture de l'accident

L'environnement `comite_lecture` est passé de 10 vecteurs à 2 — les manuscrits
en cours d'évaluation ont été écrasés par du contenu catalogue, parce que la
réindexation n'a pas reçu de cible explicite. **Aucun message d'erreur** n'a
été émis : du point de vue de la fonction, elle a fait ce qu'on lui a demandé,
écrire dans l'emplacement par défaut. Le défaut n'est pas un crash, c'est une
*perte de données silencieuse* sur une instance servant du contenu en ligne.

La leçon tient au type de la fonction : `env_cible` devrait être **obligatoire**
(un paramètre positionnel, pas un mot-clé par défaut à `None`). Un langage de
requête qui accepte « réindexer » sans dire « où » laisse ouverte la porte de
l'écrasement. Lever l'ambiguïté à la signature, pas à la documentation.


## 7. Exercices

Les trois exercices suivants manipulent le même vector store synthétique. Les
stub sont à compléter — `return None` ou `pass`.


### Exercice 1 — le taux de fuite d'un chatbot donné

Écrire une fonction qui, pour un chatbot donné (son environnement autorisé) et
une liste de requêtes, renvoie le taux de fuite global **sans filtre** puis
**avec filtre**. Le but est de produire le tableau comparatif pour un seul
chatbot plutôt que sur la batterie agrégée.


In [7]:
def taux_fuite_chatbot(store, env_autorise, requetes, k=5):
    """Renvoie (sans_filtre, avec_filtre) -- deux taux de fuite.

    sans_filtre : retrieval sur TOUS les environnements.
    avec_filtre : retrieval sur env_autorise uniquement.
    """
    # TODO : calculer les deux taux sur la liste de requetes.
    return None, None


### Exercice 2 — diagnostiquer la fonction `reindexer`

Écrire une fonction `detecter_ecrasement` qui compare l'état du store avant et
après un appel à `reindexer`, et renvoie le nom de l'environnement écrasé (ou
`None` si rien n'a été perdu).


In [8]:
def detecter_ecrasement(store_avant, store_apres):
    """Renvoie le nom de l'environnement dont le nombre de vecteurs a
    diminue apres une operation, ou None si aucun n'a perdu de vecteurs.
    """
    # TODO : comparer les deux stores et trouver l'environnement ampute.
    pass


### Exercice 3 — quels environnements pour quel chatbot ?

Un chatbot « comité de lecture » doit pouvoir lire les manuscrits soumis
(`comite_lecture`) **et** le catalogue déjà publié (`catalogue_public`) pour
comparer. Écrire une fonction `plus_proches_multi_envs` qui étend le retrieval
à une **liste** d'environnements autorisés (pas un seul). C'est le cas réel :
un chatbot sert un rôle qui cumule plusieurs régimes d'accès.


In [9]:
def plus_proches_multi_envs(store, requete, k, envs_autorises):
    """Renvoie les k chunks les plus proches, en ne cherchant que dans
    la liste d'environnements autorises (pas un seul, plusieurs).
    """
    # TODO : etendre plus_proches a un ensemble d'environnements.
    return None


## 8. Provenance et limites

**Ce que ce notebook mesure.** La *structure* d'un défaut d'accès : un vector
store non partitionné laisse fuiter le contenu entre régimes, et un langage de
réindexation ambigu écrase silencieusement un voisin. Les deux sont
déterministes sur données synthétiques.

**Ce qu'il ne mesure pas.** La qualité du retrieval (précision, rappel), la
latence, le coût par requête. Ces grandeurs dépendent du modèle d'embeddings,
du volume du corpus et du provider ; les publier depuis un fixture synthétique
n'aurait aucun sens.

**Pour aller plus loin.**
- [`ingestion-corpus-long-rag.ipynb`](ingestion-corpus-long-rag.ipynb) —
  l'amont : comment découper un corpus avant de l'indexer.
- [`auditer-un-serveur-mcp.ipynb`](auditer-un-serveur-mcp.ipynb) — l'autre
  moitié du contrôle d'accès : ce qu'un **agent** peut appeler (catalogue
  d'outils), pas seulement ce qu'un chatbot peut **lire** (vector store).
- [`livresagites-parcours.md`](livresagites-parcours.md) Parcours 2 — le cas
  d'usage réel dont ce notebook est l'illustration exécutable.
